<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 04. GPT-1

## Improving Language Understanding by Generative Pre-Training

- **원 논문:** [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- **저자 / 발표:** Alec Radford, Karthik Narasimhan, Tim Salimans, Ilya Sutskever · OpenAI technical report (2018)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** causal LM pretraining과 supervised+auxiliary LM fine-tuning을 재현한다.

**축소하거나 생략한 부분:** 12-layer decoder와 BooksCorpus 대신 1-block decoder와 360개 행을 사용한다. 로컬 corpus는 6개 문장 template을 반복하므로 240/120 row split은 입력 행을 격리하지만 새로운 언어 분포의 일반화를 뜻하지 않는다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2 Eq. (1): L1(U) autoregressive LM | Task 2–3 | LM loss |
| §2 Eq. (2): Transformer decoder | Task 1–2 | causal mask |
| §2 Eq. (3): supervised objective L2 | Task 3 classifier | CE loss |
| §2 Eq. (4), Fig. 1: L2+λL1 | Task 3 | 두 loss plot |

## 핵심 아이디어와 수식

$$L_1(U)=\sum_i\log P(u_i\mid u_{i-k},\ldots,u_{i-1}),\quad L_3=L_2+\lambda L_1$$

**기호 해설:** L1은 causal LM, L2는 supervised loss, λ는 fine-tuning 중 LM 보조 가중치다.

**코드 대응:** Task 1이 causal mask, Task 2가 decoder/두 head, Task 3이 pretrain→fine-tune을 구현한다.

**입력과 출력 shape:** LM input [B,4] → hidden [B,4,24] → LM [B,4,V], classifier [B,2].

**포트폴리오 구현 범위:** 대규모 corpus와 12-layer는 축소하지만 두 단계 objective와 causal 제약은 보존한다. 여섯 template 반복이라는 holdout 한계도 함께 기록한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Eq. (1): L1(U) autoregressive LM
- **노트북에서 구현할 부분:** Task 2–3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** LM loss

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class GPTConfig:
    model_dim: int = 24
    heads: int = 4
    feedforward_dim: int = 48
    auxiliary_weight: float = 0.2
    pretrain_steps: int = 14
    finetune_steps: int = 12
    learning_rate: float = 0.02


def causal_attention_mask(length, device):
    positions = torch.arange(length, device=device)
    return positions[None, None, :, None] >= positions[None, None, None, :]


def shift_right(tokens, bos_id):
    bos = tokens.new_full((len(tokens), 1), bos_id)
    return torch.cat((bos, tokens[:, :-1]), dim=1)


config = GPTConfig()
downstream_train_index = torch.arange(
    240,
    device=sentence_ids.device,
)
downstream_test_index = torch.arange(
    240,
    len(sentence_ids),
    device=sentence_ids.device,
)
pretrain_index = downstream_train_index.clone()
assert not bool(torch.isin(pretrain_index, downstream_test_index).any())
assert not bool(
    torch.isin(
        downstream_train_index,
        downstream_test_index,
    ).any()
)
pretrain_tokens = sentence_ids[pretrain_index]
finetune_tokens = sentence_ids[downstream_train_index]
evaluation_tokens = sentence_ids[downstream_test_index]
pretrain_inputs = shift_right(pretrain_tokens, BOS)
finetune_inputs = shift_right(finetune_tokens, BOS)
evaluation_inputs = shift_right(evaluation_tokens, BOS)
finetune_labels = sentiment_labels[downstream_train_index]
evaluation_labels = sentiment_labels[downstream_test_index]
causal_mask = causal_attention_mask(sentence_ids.shape[1], DEVICE)
assert causal_mask.shape == (1, 1, 4, 4)
assert not bool(causal_mask[0, 0, 0, 1])
assert pretrain_inputs.shape == finetune_inputs.shape == (240, 4)
assert evaluation_inputs.shape == (120, 4)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Eq. (2): Transformer decoder
- **노트북에서 구현할 부분:** Task 1–2
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** causal mask

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = config.heads
        self.head_dim = config.model_dim // config.heads
        self.qkv = nn.Linear(config.model_dim, config.model_dim * 3)
        self.output = nn.Linear(config.model_dim, config.model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.view(batch, length, self.heads, self.head_dim)
        return tensor.transpose(1, 2)

    def forward(self, hidden):
        query, key, value = self.qkv(hidden).chunk(3, dim=-1)
        query = self.split_heads(query)
        key = self.split_heads(key)
        value = self.split_heads(value)
        mask = causal_attention_mask(hidden.shape[1], hidden.device)
        scores = torch.matmul(query, key.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)
        weights = scores.masked_fill(~mask, float("-inf")).softmax(dim=-1)
        attended = torch.matmul(weights, value).transpose(1, 2).contiguous()
        attended = attended.view(len(hidden), hidden.shape[1], -1)
        return self.output(attended), weights


class GPT1Mini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.model_dim)
        self.position = nn.Embedding(4, config.model_dim)
        self.attention = CausalSelfAttention(config)
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.GELU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.lm_head = nn.Linear(config.model_dim, vocab_size)
        self.classifier = nn.Linear(config.model_dim, 2)

    def hidden_states(self, token_ids):
        positions = torch.arange(token_ids.shape[1], device=token_ids.device)
        hidden = self.embedding(token_ids) + self.position(positions)[None]
        attended, _ = self.attention(hidden)
        hidden = self.norm1(hidden + attended)
        return self.norm2(hidden + self.mlp(hidden))

    def forward(self, token_ids):
        hidden = self.hidden_states(token_ids)
        return self.lm_head(hidden), self.classifier(hidden[:, -1])


model = GPT1Mini(VOCAB_SIZE, config).to(DEVICE)
lm_logits, class_logits = model(pretrain_inputs)
assert lm_logits.shape == (240, 4, VOCAB_SIZE)
assert class_logits.shape == (240, 2)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Eq. (4), Fig. 1: L2+λL1
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** 두 loss plot

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def language_model_loss(logits, targets):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
    )


def pretrain_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    lm_logits, _ = model(inputs)
    loss = language_model_loss(lm_logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def finetune_step(model, optimizer, inputs, tokens, labels, config):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    lm_logits, class_logits = model(inputs)
    supervised = F.cross_entropy(class_logits, labels)
    auxiliary = language_model_loss(lm_logits, tokens)
    total = supervised + config.auxiliary_weight * auxiliary
    total.backward()
    optimizer.step()
    return float(supervised.detach())


def evaluate_gpt(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        _, class_logits = model(inputs)
    return float((class_logits.argmax(1) == labels).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
pretrain_losses = []
for step in range(config.pretrain_steps):
    pretrain_losses.append(
        pretrain_step(
            model,
            optimizer,
            pretrain_inputs,
            pretrain_tokens,
        )
    )
finetune_losses = []
for step in range(config.finetune_steps):
    finetune_losses.append(
        finetune_step(
            model,
            optimizer,
            finetune_inputs,
            finetune_tokens,
            finetune_labels,
            config,
        )
    )
accuracy = evaluate_gpt(
    model,
    evaluation_inputs,
    evaluation_labels,
)
assert min(pretrain_losses[3:]) < pretrain_losses[0]
assert min(finetune_losses[3:]) < finetune_losses[0]
plt.figure(figsize=(6, 3))
plt.plot(pretrain_losses, label="LM pretrain")
offset = len(pretrain_losses)
x_axis = range(offset, offset + len(finetune_losses))
plt.plot(x_axis, finetune_losses, label="supervised")
plt.title(f"GPT-1 held-out row accuracy={accuracy:.1%}")
plt.legend()
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 12-layer decoder와 BooksCorpus 대신 1-block decoder와 360개 행을 사용한다. 로컬 corpus는 6개 문장 template을 반복하므로 240/120 row split은 입력 행을 격리하지만 새로운 언어 분포의 일반화를 뜻하지 않는다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.